In [ ]:
# data_preprocess.py

import pandas as pd


def read_data(train_file="dataset/train.tsv"):
    train_df = pd.read_csv(train_file, sep="\t")
    return train_df["Phrase"].values, train_df["Sentiment"].values


if __name__ == "__main__":
    X_data, y_data = read_data()
    print("train size", len(X_data))


In [ ]:
# feature_extraction.py

import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords", quiet=True)


_SENTIMENT_CRITICAL = {
    "not",
    "no",
    "nor",
    "never",
    "nothing",
    "nobody",
    "nowhere",
    "neither",
    "very",
    "too",
    "so",
    "more",
    "most",
    "less",
    "least",
    "quite",
    "rather",
    "really",
    "truly",
    "just",
    "only",
}

_base_stopwords = set(stopwords.words("english"))


STOP_WORDS = _base_stopwords - _SENTIMENT_CRITICAL

STEMMER = PorterStemmer()


def clean_text(text):
    """
    Standardizes text by lowercasing, removing punctuation,
    removing stop words, and stemming the core words.
    Sentiment-critical words (not, very, never, no, etc.) are intentionally
    kept so the model can distinguish "good" from "not good".
    """
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    words = text.split()
    cleaned_words = [STEMMER.stem(w) for w in words if w not in STOP_WORDS]
    return cleaned_words


def l2_normalize(matrix):
    """
    L2-normalize each row so that document length doesn't dominate the
    feature magnitude. Short and long reviews become comparable.
    """
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)
    return matrix / norms


class BagOfWord:
    def __init__(self, min_freq=3):
        self.vocab = {}
        self.min_freq = min_freq

    def fit(self, sent_list):
        word_counts = {}
        for sent in sent_list:
            for word in clean_text(sent):
                word_counts[word] = word_counts.get(word, 0) + 1
        for word, count in word_counts.items():
            if count >= self.min_freq:
                self.vocab[word] = len(self.vocab)

    def transform(self, sent_list, normalize=True):
        feat = np.zeros((len(sent_list), len(self.vocab)), dtype=np.float32)
        for idx, sent in enumerate(sent_list):
            for word in clean_text(sent):
                if word in self.vocab:
                    feat[idx][self.vocab[word]] += 1
        return l2_normalize(feat) if normalize else feat

    def fit_transform(self, sent_list, normalize=True):
        self.fit(sent_list)
        return self.transform(sent_list, normalize)


class TFIDF:
    def __init__(self, min_freq=3):
        self.vocab = {}
        self.idf = {}
        self.min_freq = min_freq

    def fit(self, sent_list):
        word_counts = {}
        for sent in sent_list:
            for word in clean_text(sent):
                word_counts[word] = word_counts.get(word, 0) + 1
        for word, count in word_counts.items():
            if count >= self.min_freq:
                self.vocab[word] = len(self.vocab)

        N = len(sent_list)
        doc_freq = {word: 0 for word in self.vocab}
        for sent in sent_list:
            for word in set(clean_text(sent)):
                if word in self.vocab:
                    doc_freq[word] += 1

        self.idf = {word: np.log(N / (df + 1)) for word, df in doc_freq.items()}

    def transform(self, sent_list, normalize=True):
        feat = np.zeros((len(sent_list), len(self.vocab)), dtype=np.float32)
        for idx, sent in enumerate(sent_list):
            words = clean_text(sent)
            doc_len = max(len(words), 1)
            doc_tf = {}
            for word in words:
                if word in self.vocab:
                    doc_tf[word] = doc_tf.get(word, 0) + 1
            for word, count in doc_tf.items():
                feat[idx][self.vocab[word]] = (count / doc_len) * self.idf[word]
        return l2_normalize(feat) if normalize else feat

    def fit_transform(self, sent_list, normalize=True):
        self.fit(sent_list)
        return self.transform(sent_list, normalize)


class NGram:
    def __init__(self, ngram, min_freq=3, binary=False):
        self.ngram = ngram
        self.feature_map = {}
        self.min_freq = min_freq
        self.binary = binary

    def fit(self, sent_list):
        gram_counts = {}
        for gram_n in self.ngram:
            for sent in sent_list:
                words = clean_text(sent)
                for i in range(len(words) - gram_n + 1):
                    feature = "_".join(words[i : i + gram_n])
                    gram_counts[feature] = gram_counts.get(feature, 0) + 1
        for feature, count in gram_counts.items():
            if count >= self.min_freq:
                self.feature_map[feature] = len(self.feature_map)

    def transform(self, sent_list, normalize=True):
        feat = np.zeros((len(sent_list), len(self.feature_map)), dtype=np.float32)
        for idx, sent in enumerate(sent_list):
            words = clean_text(sent)
            for gram_n in self.ngram:
                for i in range(len(words) - gram_n + 1):
                    feature = "_".join(words[i : i + gram_n])
                    if feature in self.feature_map:
                        if self.binary:
                            feat[idx][self.feature_map[feature]] = 1
                        else:
                            feat[idx][self.feature_map[feature]] += 1
        return l2_normalize(feat) if normalize else feat

    def fit_transform(self, sent_list, normalize=True):
        self.fit(sent_list)
        return self.transform(sent_list, normalize)


In [ ]:
# softmax_regression.py

import numpy as np
from scipy.sparse import issparse, hstack, csr_matrix


def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


def compute_class_weights(y_flat, num_classes):
    """
    Inverse-frequency weights so rare classes get proportionally larger gradients.
    Formula: w_c = total / (num_classes * count_c)
    """
    weights = np.zeros(num_classes)
    total = len(y_flat)
    for c in range(num_classes):
        count = np.sum(y_flat == c)
        if count > 0:
            weights[c] = total / (num_classes * count)
    return weights


def _row_slice(X, indices):
    """Slice rows from either a sparse or dense matrix."""
    if issparse(X):
        return X[indices]
    return X[indices]


def _dot(X, W_T):
    """
    X @ W.T — works for both dense numpy arrays and scipy sparse matrices.
    Returns a dense numpy array (softmax always needs dense output).
    """
    if issparse(X):
        return np.asarray(X.dot(W_T))
    return X.dot(W_T)


def _add_bias(X):
    """
    Prepend a column of ones (bias trick) for both sparse and dense inputs.
    For sparse matrices this avoids converting the whole matrix to dense.
    """
    if issparse(X):
        n = X.shape[0]
        bias_col = csr_matrix(np.ones((n, 1), dtype=np.float32))
        return hstack([bias_col, X], format="csr")
    return np.c_[np.ones((X.shape[0], 1), dtype=np.float32), X]


class SoftmaxRegression:
    def __init__(self):
        self.num_of_class = None
        self.weight = None

    def fit(
        self,
        X_raw,
        y,
        learning_rate=0.1,
        epoch=30,
        num_of_class=3,
        print_loss_steps=5,
        update_strategy="mini_batch",
        lambda_reg=0.001,
        batch_size=256,
        lr_decay=0.98,
        use_class_weights=True,
    ):
        X = _add_bias(X_raw)
        self.n = X.shape[0]
        self.m = X.shape[1]
        self.num_of_class = num_of_class

        y_flat = np.array(y).flatten().astype(int)

        if use_class_weights:
            class_weights = compute_class_weights(y_flat, num_of_class)
            print(
                f"  Class weights: { {i: round(w, 2) for i, w in enumerate(class_weights)} }"
            )
        else:
            class_weights = np.ones(num_of_class)

        self.weight = (
            np.random.randn(self.num_of_class, self.m).astype(np.float32) * 0.01
        )

        y_one_hot = np.zeros((self.n, self.num_of_class), dtype=np.float32)
        y_one_hot[np.arange(self.n), y_flat] = 1.0

        current_lr = float(learning_rate)
        loss_history = []

        for e in range(epoch):
            rand_index = np.random.permutation(self.n)
            epoch_loss = 0.0

            if update_strategy == "stochastic":
                for index in rand_index:
                    Xi = _row_slice(X, [index])
                    prob = softmax(_dot(Xi, self.weight.T)).flatten()
                    c = y_flat[index]
                    epoch_loss += class_weights[c] * -np.log(prob[c] + 1e-15)

                    grad = class_weights[c] * (y_one_hot[index] - prob)

                    if issparse(Xi):
                        weight_update = np.outer(grad, Xi.toarray().flatten())
                    else:
                        weight_update = np.outer(grad, Xi.flatten())
                    self.weight += current_lr * weight_update
                    self.weight[:, 1:] *= 1.0 - current_lr * lambda_reg

            elif update_strategy == "mini_batch":
                for start in range(0, self.n, batch_size):
                    idx = rand_index[start : start + batch_size]
                    Xi = _row_slice(X, idx)
                    yi = y_flat[idx]
                    yi_oh = y_one_hot[idx]

                    prob = softmax(_dot(Xi, self.weight.T))

                    sample_weights = class_weights[yi]
                    correct_probs = prob[np.arange(len(idx)), yi]
                    epoch_loss += float(
                        np.sum(sample_weights * -np.log(correct_probs + 1e-15))
                    )

                    grad = (yi_oh - prob) * sample_weights[:, None]
                    grad /= len(idx)

                    if issparse(Xi):
                        weight_update = np.asarray(grad.T.dot(Xi.toarray()))
                    else:
                        weight_update = grad.T.dot(Xi)

                    self.weight += current_lr * weight_update
                    self.weight[:, 1:] *= 1.0 - current_lr * lambda_reg

            current_lr *= lr_decay

            l2_penalty = (lambda_reg / 2.0) * float(np.sum(self.weight[:, 1:] ** 2))
            loss = float(epoch_loss / self.n) + l2_penalty
            loss_history.append(loss)

            if print_loss_steps > 0 and (e % print_loss_steps == 0 or e == epoch - 1):
                print(f"  epoch {e:3d} | loss {loss:.4f} | lr {current_lr:.5f}")

        return loss_history

    def predict(self, X_raw):
        X = _add_bias(X_raw)
        prob = softmax(_dot(X, self.weight.T))
        return prob.argmax(axis=1)

    def predict_proba(self, X_raw):
        X = _add_bias(X_raw)
        return softmax(_dot(X, self.weight.T))

    def score(self, X, y):
        pred = self.predict(X)
        y_flat = np.array(y).flatten()
        return float(np.sum(pred == y_flat) / len(y_flat))


In [ ]:
# main.py

import numpy as np
# from data_preprocess import read_data
# from feature_extraction import TFIDF, NGram
# from softmax_regression import SoftmaxRegression
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix

if __name__ == "__main__":

    debug = 1
    X_data, y_data = read_data()

    if debug == 1:
        X_data = X_data[:50000]
        y_data = y_data[:50000]

    y = np.array(y_data).flatten().astype(int)

    y_mapped = np.zeros_like(y)
    y_mapped[(y == 0) | (y == 1)] = 0
    y_mapped[y == 2] = 1
    y_mapped[(y == 3) | (y == 4)] = 2
    y = y_mapped

    print("Dataset label distribution:")
    names = ["Negative", "Neutral ", "Positive"]
    for c in range(3):
        count = int(np.sum(y == c))
        bar = "█" * int(count / len(y) * 40)
        print(f"  {names[c]} ({c}): {count:6d} ({count/len(y)*100:4.1f}%)  {bar}")
    print()

    tfidf_model = TFIDF(min_freq=5)
    ngram_model = NGram(ngram=(1, 2), min_freq=5, binary=False)

    print("Extracting TF-IDF features...")
    X_Tfidf_dense = tfidf_model.fit_transform(X_data, normalize=True)
    X_Tfidf = csr_matrix(X_Tfidf_dense)
    del X_Tfidf_dense
    print(
        f"  TF-IDF shape: {X_Tfidf.shape}  "
        f"density: {X_Tfidf.nnz / (X_Tfidf.shape[0]*X_Tfidf.shape[1])*100:.2f}%  "
        f"memory: ~{X_Tfidf.data.nbytes/1e6:.0f} MB"
    )

    print("Extracting N-Gram features...")
    X_Gram_dense = ngram_model.fit_transform(X_data, normalize=True)
    X_Gram = csr_matrix(X_Gram_dense)
    del X_Gram_dense
    print(
        f"  N-Gram  shape: {X_Gram.shape}  "
        f"density: {X_Gram.nnz / (X_Gram.shape[0]*X_Gram.shape[1])*100:.2f}%  "
        f"memory: ~{X_Gram.data.nbytes/1e6:.0f} MB"
    )

    X_tr_tfidf, X_te_tfidf, y_tr, y_te = train_test_split(
        X_Tfidf, y, test_size=0.2, random_state=42, stratify=y
    )
    X_tr_gram, X_te_gram, y_tr_g, y_te_g = train_test_split(
        X_Gram, y, test_size=0.2, random_state=42, stratify=y
    )

    del X_Tfidf, X_Gram

    epoch = 40
    learning_rate = 0.3
    lambda_reg = 0.0005
    lr_decay = 0.98
    batch_size = 256

    print("\n--- Training TF-IDF Model ---")
    model1 = SoftmaxRegression()
    history1 = model1.fit(
        X_tr_tfidf,
        y_tr,
        num_of_class=3,
        epoch=epoch,
        learning_rate=learning_rate,
        lambda_reg=lambda_reg,
        update_strategy="mini_batch",
        batch_size=batch_size,
        lr_decay=lr_decay,
        print_loss_steps=5,
        use_class_weights=True,
    )

    plt.figure()
    plt.plot(history1)
    plt.title("TF-IDF Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.tight_layout()
    plt.show()

    print(
        f"TF-IDF  train: {model1.score(X_tr_tfidf, y_tr):.4f} | "
        f"test:  {model1.score(X_te_tfidf, y_te):.4f}"
    )
    preds1 = model1.predict(X_te_tfidf)
    print("  Per-class accuracy:")
    for c in range(3):
        mask = y_te == c
        if mask.sum() > 0:
            acc = np.mean(preds1[mask] == c)
            print(f"    {names[c]}: {acc:.3f}  (n={mask.sum()})")

    print("\n--- Training N-Gram Model ---")
    model2 = SoftmaxRegression()
    history2 = model2.fit(
        X_tr_gram,
        y_tr_g,
        num_of_class=3,
        epoch=epoch,
        learning_rate=learning_rate,
        lambda_reg=lambda_reg,
        update_strategy="mini_batch",
        batch_size=batch_size,
        lr_decay=lr_decay,
        print_loss_steps=5,
        use_class_weights=True,
    )

    plt.figure()
    plt.plot(history2)
    plt.title("N-Gram Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.tight_layout()
    plt.show()

    print(
        f"N-Gram  train: {model2.score(X_tr_gram, y_tr_g):.4f} | "
        f"test:  {model2.score(X_te_gram, y_te_g):.4f}"
    )
    preds2 = model2.predict(X_te_gram)
    print("  Per-class accuracy:")
    for c in range(3):
        mask = y_te_g == c
        if mask.sum() > 0:
            acc = np.mean(preds2[mask] == c)
            print(f"    {names[c]}: {acc:.3f}  (n={mask.sum()})")


In [ ]:
# 6. Inference
print("\n" + "=" * 40)
print("AI SENTIMENT PREDICTOR")
print("=" * 40)

sentiment_map = {
    0: "Negative 🙁",
    1: "Neutral 😐",
    2: "Positive 🙂",
}

while True:
    user_input = input("\nEnter a movie review (or 'exit' to quit): ").strip()
    if user_input.lower() == "exit":
        break
    if not user_input:
        continue

    phrase = [user_input]
    # transform() returns dense; convert to sparse for the model
    tfidf_feat = csr_matrix(tfidf_model.transform(phrase, normalize=True))
    gram_feat = csr_matrix(ngram_model.transform(phrase, normalize=True))

    p1 = model1.predict(tfidf_feat)[0]
    p2 = model2.predict(gram_feat)[0]
    proba1 = model1.predict_proba(tfidf_feat)[0]
    proba2 = model2.predict_proba(gram_feat)[0]

    bar1 = "  ".join(f"{names[i].strip()}:{proba1[i]*100:.0f}%" for i in range(3))
    bar2 = "  ".join(f"{names[i].strip()}:{proba2[i]*100:.0f}%" for i in range(3))
    print(f"\n  TF-IDF -> {sentiment_map[p1]:22s}  conf:{proba1[p1]*100:.1f}%")
    print(f"    {bar1}")
    print(f"  N-Gram -> {sentiment_map[p2]:22s}  conf:{proba2[p2]*100:.1f}%")
    print(f"    {bar2}")